# Experiment 3.1: controlled fine-tuning from Experiment 3

This notebook continues from the final Experiment 3 policy. It preserves the learned dual-scale CNN-DQN, reward coefficients, action masking, and oscillation/stationary deadlocks while changing only initialization, the no-progress timeout, and the epsilon schedule. Experiment 1, 2, and 3 outputs are never overwritten.

## Final Experiment 3 artifact diagnosis

The episode-2000 GIFs, action logs, mapper snapshots, and trajectory curves were inspected for the three fixed diagnostic seeds.

| Seed | Leaves initial room | Crosses narrow gap | Multiple rooms | Failure behavior | Final map coverage |
|---:|:---:|:---:|:---:|---|---:|
| 20000 | No | No | No | Stops advancing near a reachable frontier (distance 4), then alternates left/right until the unchanged 8-step oscillation deadlock fires. | 21.6% |
| 20050 | No | No | No | Stops advancing near a reachable frontier (distance 3), then alternates left/right until the unchanged 8-step oscillation deadlock fires. | 26.3% |
| 20099 | Yes | Yes | Yes | Successfully crosses a one-cell wall gap and explores multiple rooms. Later it travels through known space without discovery; frontier distance varies and the old 38-step discovery-only timeout fires at step 140. It is neither oscillating nor stationary at termination. | 58.2% |

**Diagnostic answer:** Experiment 3 has partially learned gap crossing: seed 20099 proves that it can deliberately pass through a narrow opening and continue into another room. The behavior is not consistent because two seeds fail inside the first room, and the successful trajectory is later cut off by the discovery-only timeout while moving through known space. Experiment 3.1 therefore tests the stated hypothesis without using this diagnosis to retune the model or rewards.

## Experiment 3.1 setup

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import torch

from utils import alert_training_complete
from utils.minigrid_dual_scale_dqn import DualScaleReplayBuffer, DualScaleState
from utils.minigrid_experiment3 import load_experiment3_policy
from utils.minigrid_experiment3_1 import (
    Experiment31DeadlockConfig,
    Experiment3RewardConfig,
    ExplorationProgressDetector,
    episode_linear_epsilon,
    evaluate_exploration_controller,
    initialize_experiment_3_1,
    load_experiment31_policy,
    run_experiment_3_1,
    save_comparison_artifacts,
    summarize_evaluations,
    transition_is_done,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

experiment_3_model_path = Path("experiment_3/final_model.pth")
experiment_3_1_output_dir = Path("experiment_3_1")
experiment_3_1_model_path = experiment_3_1_output_dir / "final_model.pth"

RUN_EXPERIMENT_3_1 = False
RUN_GREEDY_EVALUATION = False
RUN_MATCHED_COMPARISON = False

if not experiment_3_model_path.exists():
    raise FileNotFoundError(experiment_3_model_path.resolve())

In [ ]:
source_checkpoint = torch.load(
    experiment_3_model_path, map_location="cpu", weights_only=False
)
source_hyperparameters = source_checkpoint["hyperparameters"]
tensor_config = dict(source_checkpoint["tensor_config"])
training_environment_config = dict(source_checkpoint["training_environment_config"])
reward_config = Experiment3RewardConfig(**source_checkpoint["reward_config"])
deadlock_config = Experiment31DeadlockConfig(
    oscillation_deadlock_threshold=source_checkpoint["deadlock_config"]["oscillation_deadlock_threshold"],
    stationary_deadlock_threshold=source_checkpoint["deadlock_config"]["stationary_deadlock_threshold"],
    deadlock_penalty=source_checkpoint["deadlock_config"]["deadlock_penalty"],
    no_progress_timeout_multiplier=2.0,
)

training_seed_start = source_hyperparameters["training_seed_range"][0]
training_seed_count = (
    source_hyperparameters["training_seed_range"][1] - training_seed_start + 1
)
validation_seeds = list(range(20_000, 20_100))
validation_subset = validation_seeds[:20]
diagnostic_validation_seeds = [20_000, 20_050, 20_099]

print("Inherited architecture:", source_checkpoint["architecture"])
print("Inherited reward:", source_checkpoint["reward_config"])
print("No-progress timeout (19x19):", deadlock_config.no_progress_timeout(19, 19))

## Focused smoke tests

In [ ]:
# Inheritance, fresh optimizer/replay, and synchronized target.
policy_net, target_net, optimizer, replay_buffer, loaded_source = initialize_experiment_3_1(
    experiment_3_model_path,
    device,
    source_hyperparameters["learning_rate"],
    source_hyperparameters["replay_capacity"],
)
assert len(replay_buffer) == 0
assert len(optimizer.state) == 0
for name, inherited in loaded_source["model_state_dict"].items():
    assert torch.equal(policy_net.state_dict()[name].cpu(), inherited.cpu())
for policy_value, target_value in zip(policy_net.state_dict().values(), target_net.state_dict().values()):
    assert torch.equal(policy_value, target_value)

# The controlled reward and unchanged deadlock parameters come from the final checkpoint.
assert reward_config.novelty_beta == loaded_source["reward_config"]["novelty_beta"]
assert reward_config.frontier_progress_beta == loaded_source["reward_config"]["frontier_progress_beta"]
assert reward_config.stationary_penalty == loaded_source["reward_config"]["stationary_penalty"]
assert reward_config.oscillation_penalty == loaded_source["reward_config"]["oscillation_penalty"]
assert reward_config.blocked_penalty == loaded_source["reward_config"]["blocked_penalty"]
assert deadlock_config.oscillation_deadlock_threshold == loaded_source["deadlock_config"]["oscillation_deadlock_threshold"]
assert deadlock_config.stationary_deadlock_threshold == loaded_source["deadlock_config"]["stationary_deadlock_threshold"]
assert deadlock_config.deadlock_penalty == loaded_source["deadlock_config"]["deadlock_penalty"]
assert deadlock_config.no_progress_timeout(19, 19) == 76

# Episode-based epsilon has exact endpoints and is independent of environment steps.
assert np.isclose(episode_linear_epsilon(0, 1001), 0.10)
assert np.isclose(episode_linear_epsilon(500, 1001), 0.06)
assert episode_linear_epsilon(1000, 1001) == 0.02

def detector_step(detector, before, after, new_cells=0, position_changed=True):
    return detector.update(
        action=2, previous_action=2, position_changed=position_changed,
        new_cells=new_cells, frontier_distance_before=before,
        frontier_distance_after=after,
    )

# Discovery and reduced reachable-frontier distance both reset progress.
detector = ExplorationProgressDetector(deadlock_config, no_progress_timeout=76)
for _ in range(10):
    detector_step(detector, 12, 12)
assert detector.steps_since_exploration_progress == 10
status = detector_step(detector, 12, 13, new_cells=1)
assert status["exploration_progress_this_step"] and status["steps_since_exploration_progress"] == 0
for _ in range(10):
    detector_step(detector, 12, 12)
status = detector_step(detector, 12, 11)
assert status["frontier_progress_this_step"] and status["steps_since_exploration_progress"] == 0

# More than width+height steps toward a frontier must not terminate.
detector = ExplorationProgressDetector(deadlock_config, no_progress_timeout=76)
for distance in range(100, 59, -1):
    status = detector_step(detector, distance, distance - 1)
    assert status["deadlock_type"] is None
assert detector.steps_since_exploration_progress == 0

# No discovery and no net frontier progress terminates exactly at the timeout.
detector = ExplorationProgressDetector(deadlock_config, no_progress_timeout=76)
for step in range(76):
    status = detector_step(detector, 12, 12)
    if step < 75:
        assert status["deadlock_type"] is None
assert status["deadlock_type"] == "no_progress_deadlock"

# The final bad transition remains in replay with done=True.
dummy = DualScaleState(
    local=np.zeros((4, 15, 15), dtype=np.float32),
    global_map=np.zeros((5, 15, 15), dtype=np.float32),
    global_scale_log2=0.0,
)
terminal_buffer = DualScaleReplayBuffer(2)
terminal_buffer.push(dummy, 2, deadlock_config.deadlock_penalty, dummy, transition_is_done(False, False, status["deadlock_type"]))
assert terminal_buffer.buffer[-1][-1] is True

print("Inheritance, controlled configuration, timeout, epsilon, and replay checks passed.")

In [ ]:
# One short greedy CUDA episode verifies evaluation and GIF/checkpoint-adjacent artifacts.
smoke_environment_config = dict(training_environment_config)
smoke_environment_config["max_steps"] = 4
with TemporaryDirectory() as temporary_directory:
    smoke_results = evaluate_exploration_controller(
        [20_000],
        smoke_environment_config,
        tensor_config,
        device,
        reward_config,
        deadlock_config=deadlock_config,
        policy_net=policy_net,
        artifact_dir=Path(temporary_directory),
        validation_checkpoint_episode=0,
    )
    smoke_summary = summarize_evaluations(smoke_results)
    assert smoke_summary["random_action_count"] == 0
    assert (Path(temporary_directory) / "seed_20000.gif").exists()
    assert (Path(temporary_directory) / "seed_20000_actions.json").exists()
print("Greedy validation and diagnostic artifact smoke test passed on", device)

## Fine-tuning configuration

In [ ]:
num_experiment_3_1_episodes = 1_000
gamma = source_hyperparameters["gamma"]
learning_rate = source_hyperparameters["learning_rate"]
batch_size = source_hyperparameters["batch_size"]
replay_capacity = source_hyperparameters["replay_capacity"]
minimum_replay_size = source_hyperparameters["minimum_replay_size"]
target_update_frequency = source_hyperparameters["target_update_frequency"]
gradient_clip = source_hyperparameters["gradient_clip"]
epsilon_start = 0.10
epsilon_end = 0.02
validation_frequency = source_hyperparameters["validation_frequency"]
training_random_seed = 45

print("Experiment 3.1 output:", experiment_3_1_output_dir.resolve())
print("Full training remains disabled:", not RUN_EXPERIMENT_3_1)

In [ ]:
if RUN_EXPERIMENT_3_1:
    policy_net, experiment_3_1_checkpoint = run_experiment_3_1(
        pretrained_checkpoint=experiment_3_model_path,
        output_dir=experiment_3_1_output_dir,
        environment_config=training_environment_config,
        tensor_config=tensor_config,
        validation_seeds=validation_subset,
        diagnostic_validation_seeds=diagnostic_validation_seeds,
        device=device,
        reward_config=reward_config,
        deadlock_config=deadlock_config,
        num_finetune_episodes=num_experiment_3_1_episodes,
        training_seed_start=training_seed_start,
        training_seed_count=training_seed_count,
        training_random_seed=training_random_seed,
        learning_rate=learning_rate,
        gamma=gamma,
        batch_size=batch_size,
        replay_capacity=replay_capacity,
        minimum_replay_size=minimum_replay_size,
        target_update_frequency=target_update_frequency,
        gradient_clip=gradient_clip,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        validation_frequency=validation_frequency,
    )
    alert_training_complete("Experiment 3.1 fine-tuning finished.")
else:
    print("Ready. Set RUN_EXPERIMENT_3_1 = True only when you want to start fine-tuning.")

## Greedy validation and matched-seed comparison

In [ ]:
if RUN_GREEDY_EVALUATION:
    fine_tuned_policy, _ = load_experiment31_policy(experiment_3_1_model_path, device)
    experiment_3_1_results = evaluate_exploration_controller(
        validation_seeds, training_environment_config, tensor_config, device,
        reward_config, deadlock_config=deadlock_config, policy_net=fine_tuned_policy,
    )
    experiment_3_1_summary = summarize_evaluations(experiment_3_1_results)
    assert experiment_3_1_summary["random_action_count"] == 0
    print(experiment_3_1_summary)
else:
    print("Final greedy validation is disabled until fine-tuning is complete.")

In [ ]:
if RUN_MATCHED_COMPARISON:
    experiment_3_policy, _ = load_experiment3_policy(experiment_3_model_path, device)
    experiment_3_1_policy, _ = load_experiment31_policy(experiment_3_1_model_path, device)
    experiment_3_matched = evaluate_exploration_controller(
        validation_seeds, training_environment_config, tensor_config, device,
        reward_config, deadlock_config=deadlock_config, policy_net=experiment_3_policy,
    )
    experiment_3_1_matched = evaluate_exploration_controller(
        validation_seeds, training_environment_config, tensor_config, device,
        reward_config, deadlock_config=deadlock_config, policy_net=experiment_3_1_policy,
    )
    comparison = save_comparison_artifacts(
        experiment_3_1_output_dir / "matched_seed_comparison",
        {"Experiment 3 final": experiment_3_matched, "Experiment 3.1 final": experiment_3_1_matched},
        "Experiment 3 vs 3.1 matched-seed greedy coverage",
    )
    print(comparison)
else:
    print("Matched comparison remains disabled until the Experiment 3.1 final model exists.")